In [ ]:
!git clone https://github.com/Otza02/satellite-img-segmentation.git
%cd satellite-img-segmentation
%pip install -e .

## Reiniciar sesion

In [ ]:
%cd /content/satellite-img-segmentation
!unzip -q data/train.zip -d data/train

## Entrenamiento de U-Net

In [ ]:
from satelliteSegmentation.config import Config
from satelliteSegmentation.dataset import SatelliteData, spatial_train_val_split
from satelliteSegmentation.models.unet import UNet
from satelliteSegmentation.train import train_model
from satelliteSegmentation.metrics import segmentation_metrics, dice_score
from satelliteSegmentation.utils import plot_confusion_matrix, plot_bar_metrics
from satelliteSegmentation.tokenizer import Tokenizer

import torch
from torch.utils.data import DataLoader, Subset
from matplotlib import pyplot as plt
import pandas as pd
from pathlib import Path
import json
import shutil

from google.colab import files

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64

## Preparar Datos

In [ ]:
data = SatelliteData("data/train")

train_idx, val_idx = spatial_train_val_split(data, val_fraction=0.2)
train, val = Subset(data, train_idx), Subset(data, val_idx)

train_loader = DataLoader(
    train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=device == "cuda",
    persistent_workers=device == "cuda",
)

val_loader = DataLoader(
    val,
    batch_size=batch_size,
    num_workers=2,
    pin_memory=device == "cuda",
    persistent_workers=device == "cuda",
)

In [ ]:
def calc_weights(
    dataloader,
    num_classes: int,
    ignore_index: int,
):
    counts = torch.zeros(num_classes, dtype=torch.long)

    for _, masks in dataloader:
        masks = masks.flatten()
        masks = masks[masks != ignore_index]

        counts += torch.bincount(
            masks,
            minlength=num_classes,
        )

    # Solo clases que participan en el entrenamiento
    valid_counts = counts.clone()
    valid_counts[ignore_index] = 0

    # Ejemplo: inverse square root frequency
    weights = torch.zeros(num_classes, dtype=torch.float32)

    valid = valid_counts > 0
    weights[valid] = 1.0 / torch.sqrt(valid_counts[valid].float())

    # Normalizar para que la media de los weights válidos sea 1
    weights[valid] /= weights[valid].mean()

    # El weight de la clase ignorada es irrelevante
    weights[ignore_index] = 0.0

    return counts, weights

counts, weights = calc_weights(
    train_loader,
    num_classes=7,
    ignore_index=6,
)
weights = weights.to(device)

In [ ]:
def save_model(name: str, model: torch.nn.Module, hist: dict, config: Config):
    result_folder = Path(f"result/{name}")
    result_folder.mkdir(parents=True, exist_ok=True)
    
    torch.save(model.state_dict(), result_folder / "checkpoint.pt")
    pd.DataFrame(hist).to_csv(result_folder / "hist.csv", index=False)
    with open(result_folder / "config.json", "w") as f:
        json.dump(config.to_json(), f)
    
    return result_folder

def zip_and_download(folder: Path):
    folder = Path(folder)

    zip_path = shutil.make_archive(
        base_name=str(folder),
        format="zip",
        root_dir=folder.parent,
        base_dir=folder.name,
    )

    print(f"Descargando: {zip_path}")
    files.download(zip_path)

## Train

In [ ]:
configs = {
    "baseline": Config(device, weights=weights),
    "channels-32": Config(device, hidden_channels=(32, 64, 128, 256), bottleneck_channels=512, weights=weights),
    "kernel-5": Config(device, kernel_size=5, weights=weights),
    "lr-3e-4": Config(device, lr=3e-4, weights=weights),
    "lr-1e-5": Config(device, lr=1e-5, weights=weights),
}

In [ ]:
for name, conf in configs.items():
    torch.manual_seed(2026)
    model = UNet(conf)
    print(f"Modelo: {name} | n_params: {sum([p.numel() for p in model.parameters()]):,}")

    criterion = torch.nn.CrossEntropyLoss(conf.weights, ignore_index=6)

    model, hist = train_model(model, train_loader, val_loader, criterion, conf)
    
    result_folder = save_model(name, model, hist, conf)
    zip_and_download(result_folder)